# Feature Map Visualization

可视化 RTMDet 检测模型的中间层特征图，判断模型是否学到有用的语义特征。

**可视化层：**
- Backbone: C3 (stride 8) / C4 (stride 16) / C5 (stride 32)
- Neck (PAFPN 融合后): P3 / P4 / P5

**聚合方式（channel → 2D 空间图）：**
| 模式 | 公式 | 适合看 |
|------|------|---------|
| `l2norm` | `feat.norm(dim=0)` | 特征向量整体强度（**推荐默认**）|
| `mean` | `feat.mean(0)` | 平均响应，平滑，少噪声 |
| `max` | `feat.max(0).values` | 最强单 channel 峰值 |

**前提：** 已完成训练并有 checkpoint，已运行过 `tools/convert_yolo_to_coco.py`。

## § 0  环境检查与导入

In [ ]:
# ── 安装缺失依赖（首次运行）──────────────────────────────────────────────────
# !pip install opencv-python matplotlib  # 通常已安装

import sys, os
from pathlib import Path

# 将项目根目录加入 path，使 det_baseline 和 tools 可 import
NOTEBOOK_DIR = Path(os.getcwd())
ROOT = NOTEBOOK_DIR.parent           # det-baseline/
sys.path.insert(0, str(ROOT))

import det_baseline  # 触发 mmdet / mmpretrain / mmyolo 模块注册

# 从 tools/visualize_features.py 导入所有核心函数
from tools.visualize_features import (
    load_model,
    extract_features,
    aggregate,
    normalize_to_uint8,
    make_heatmap_overlay,
    plot_feature_overview,
    plot_channel_grid,
    plot_backbone_comparison,
    print_feature_stats,
    LAYER_KEYS,
    LAYER_LABELS,
)

import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.dpi'] = 120
%matplotlib inline

print('Python :', sys.version.split()[0])
print('Torch  :', torch.__version__)
print('Device :', 'cuda' if torch.cuda.is_available() else 'cpu (no CUDA)')

## § 1  路径配置

**修改这个 cell** 以指向你的 config / checkpoint / 图像。

In [ ]:
# ── 单模型模式 ────────────────────────────────────────────────────────────────
CONFIG_PATH     = str(ROOT / 'configs' / 'rtmdet_resnet50_dr.py')
CHECKPOINT_PATH = str(ROOT / 'work_dirs' / 'rtmdet_resnet50_dr' / 'best_coco_bbox_mAP_epoch_100.pth')
IMAGE_PATH      = str(ROOT / 'dataset-example' / 'images' / 'test' / '0_346.jpg')

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

# ── 多 backbone 对比（§ 5 使用，没有多个 checkpoint 时可忽略）──────────────────
COMPARE_MODELS = [
    # (显示名称, config路径, checkpoint路径)
    # ('ResNet-50', str(ROOT/'configs'/'rtmdet_resnet50_dr.py'),     'work_dirs/rtmdet_resnet50_dr/best_*.pth'),
    # ('Swin-T',    str(ROOT/'configs'/'rtmdet_swin_t_dr.py'),       'work_dirs/rtmdet_swin_t_dr/best_*.pth'),
    # ('PVT-v2',    str(ROOT/'configs'/'rtmdet_pvtv2_b2_dr.py'),     'work_dirs/rtmdet_pvtv2_b2_dr/best_*.pth'),
    # ('EffNet-B3', str(ROOT/'configs'/'rtmdet_efficientnet_b3_dr.py'),'work_dirs/rtmdet_efficientnet_b3_dr/best_*.pth'),
]

print('Config :', CONFIG_PATH)
print('Ckpt   :', CHECKPOINT_PATH)
print('Image  :', IMAGE_PATH)

## § 2  加载模型

In [ ]:
model = load_model(CONFIG_PATH, CHECKPOINT_PATH, device=DEVICE)

print(f'Backbone : {type(model.backbone).__name__}')
print(f'Neck     : {type(model.neck).__name__}')
print(f'Head     : {type(model.bbox_head).__name__}')
print(f'Device   : {next(model.parameters()).device}')

## § 3  加载图像 & 提取 Feature Map

In [ ]:
img_bgr = cv2.imread(IMAGE_PATH)
assert img_bgr is not None, f'Cannot read image: {IMAGE_PATH}'
print(f'Original image shape: {img_bgr.shape}  (H×W×C, BGR)')

# Forward pass: backbone + neck only (head skipped)
features, img_resized = extract_features(model, img_bgr)

# 显示原图
plt.figure(figsize=(5, 5))
plt.imshow(cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB))
plt.title(f'Input image (resized to 640×640)\n{Path(IMAGE_PATH).name}', fontsize=11)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature map 统计信息 ──────────────────────────────────────────────────────
# dead channels: 激活接近 0 的 filter，数量多 → 模型可能欠拟合或训练不稳定
print_feature_stats(features)

## § 4  六层全局概览图

2 行 × 4 列布局：
- 第 0 行：原图 | Backbone C3 | C4 | C5  
- 第 1 行：原图 | Neck P3 | P4 | P5

色深代表特征向量激活强度。热区（红色）= 模型对该位置响应强烈。

In [ ]:
# ── 切换 agg 参数，观察三种聚合方式的差异 ─────────────────────────────────────
AGG = 'l2norm'   # 可改为 'mean' 或 'max'
ALPHA = 0.55     # 热力图透明度 (0=只显示原图, 1=只显示热力图)

fig = plot_feature_overview(
    features, img_resized,
    agg=AGG,
    title=f'{Path(CONFIG_PATH).stem}',
    alpha=ALPHA,
)
plt.show()

In [ ]:
# ── 三种聚合方式对比（3行×4列）──────────────────────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(21, 13))
img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)

for row_i, agg_mode in enumerate(['l2norm', 'mean', 'max']):
    axes[row_i, 0].imshow(img_rgb)
    axes[row_i, 0].set_title(f'Original\n(agg={agg_mode})', fontsize=10, fontweight='bold')
    axes[row_i, 0].axis('off')

    # 固定显示 neck P3/P4/P5
    for col_i, (key, label) in enumerate(
        zip(['neck_p3','neck_p4','neck_p5'], ['P3 stride8','P4 stride16','P5 stride32']),
        start=1
    ):
        act = aggregate(features[key], agg_mode)
        ovl = make_heatmap_overlay(img_resized, act, alpha=0.55)
        axes[row_i, col_i].imshow(ovl)
        axes[row_i, col_i].set_title(f'Neck {label}', fontsize=10)
        axes[row_i, col_i].axis('off')

fig.suptitle('Neck P3/P4/P5  —  三种聚合方式对比', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## § 5  单层 Channel Grid

选定一层，显示激活最强的 top-K 个 channel。  
每格独立归一化（VIRIDIS colormap），子标题为该 channel 的平均绝对激活值。

**如何解读：**
- 每个 filter 响应特定的视觉模式（边缘方向、纹理频率、语义区域）
- 激活集中在病灶区域 → filter 学到了有用特征
- 激活均匀分布或随机 → filter 可能未收敛

In [ ]:
# ── 参数 ──────────────────────────────────────────────────────────────────────
TARGET_LAYER = 'neck_p4'    # 可改为 LAYER_KEYS 中任意值
TOP_K        = 32           # 展示激活最强的 N 个 channel
NCOLS        = 8            # 每行显示列数

print(f'Layer: {TARGET_LAYER}  |  Feature shape: {tuple(features[TARGET_LAYER].shape)}')
print(f'Showing top-{TOP_K} channels out of {features[TARGET_LAYER].shape[1]}')

fig = plot_channel_grid(
    features[TARGET_LAYER],
    layer_name=TARGET_LAYER,
    top_k=TOP_K,
    ncols=NCOLS,
)
plt.show()

In [ ]:
# ── Channel 激活分布分析 ──────────────────────────────────────────────────────
# 直方图：每个 channel 的 mean |activation|
# 左侧堆积 = 大量低激活 channel（欠拟合信号）
# 右侧长尾 = 少量高激活 channel（语义专业化信号）

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for ax, key in zip(axes, ['backbone_c5', 'neck_p3', 'neck_p4']):
    f = features[key].squeeze(0).numpy()          # [C, H, W]
    ch_scores = np.abs(f).mean(axis=(1, 2))       # [C]
    dead_ratio = (ch_scores < 1e-4).mean() * 100  # %

    ax.hist(ch_scores, bins=40, color='steelblue', edgecolor='white', linewidth=0.4)
    ax.axvline(ch_scores.mean(), color='tomato', linewidth=1.5, label=f'mean={ch_scores.mean():.3f}')
    ax.set_title(f'{LAYER_LABELS[key]}\n({f.shape[0]} channels  |  dead: {dead_ratio:.1f}%)', fontsize=10)
    ax.set_xlabel('mean |activation|')
    ax.set_ylabel('# channels')
    ax.legend(fontsize=9)

fig.suptitle('Channel Activation Distribution  (dead threshold = 1e-4)', fontsize=12)
plt.tight_layout()
plt.show()

## § 6  多 Backbone 横向对比

同一张图、同一层，4个 backbone 并排显示。  
色阶**统一归一化**（公平对比），颜色越红 = 该区域激活越强。

> **前提：** `COMPARE_MODELS` 中所有 checkpoint 已存在。  
> 若还没训练完，可暂时只填入2个模型，或用同一个 checkpoint 测试流程。

In [ ]:
if not COMPARE_MODELS:
    print('COMPARE_MODELS 为空，请在 § 1 的路径配置 cell 中填写 checkpoint 路径后重新运行。')
else:
    print(f'加载 {len(COMPARE_MODELS)} 个模型 …')
    loaded_models = []
    for name, cfg_path, ckpt_path in COMPARE_MODELS:
        print(f'  [{name}] {ckpt_path}')
        m = load_model(cfg_path, ckpt_path, device=DEVICE)
        loaded_models.append((name, m))

    COMPARE_LAYER = 'neck_p4'   # 修改此处选择对比层
    COMPARE_AGG   = 'l2norm'

    fig = plot_backbone_comparison(
        loaded_models, img_bgr,
        layer=COMPARE_LAYER, agg=COMPARE_AGG, alpha=0.55,
    )
    plt.show()

## § 7  保存结果到文件

上面的图都是 inline 显示，如需保存 PNG：

In [ ]:
OUT_DIR = ROOT / 'vis_output'
OUT_DIR.mkdir(exist_ok=True)
stem    = Path(IMAGE_PATH).stem
backbone_tag = type(model.backbone).__name__

# ── 保存六层概览图 ─────────────────────────────────────────────────────────────
fig_ov = plot_feature_overview(features, img_resized, agg='l2norm',
                                title=backbone_tag)
out_ov = OUT_DIR / f'{stem}_{backbone_tag}_overview.png'
fig_ov.savefig(out_ov, dpi=150, bbox_inches='tight')
plt.close(fig_ov)
print(f'Saved → {out_ov}')

# ── 保存 Channel Grid ─────────────────────────────────────────────────────────
fig_ch = plot_channel_grid(features['neck_p4'], layer_name='neck_p4', top_k=32)
out_ch = OUT_DIR / f'{stem}_{backbone_tag}_neck_p4_channels.png'
fig_ch.savefig(out_ch, dpi=150, bbox_inches='tight')
plt.close(fig_ch)
print(f'Saved → {out_ch}')

---
## 参考：快捷查看某一层某种聚合

最小化代码，直接查看任意层：

In [ ]:
# ── 快速查看单层 ───────────────────────────────────────────────────────────────
layer = 'neck_p3'    # 改这里
agg   = 'l2norm'     # 改这里

act = aggregate(features[layer], agg)
ovl = make_heatmap_overlay(img_resized, act, alpha=0.6)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original', fontsize=11)
axes[0].axis('off')
axes[1].imshow(ovl)
axes[1].set_title(f'{LAYER_LABELS[layer]}  |  agg={agg}\n'
                   f'shape={tuple(features[layer].shape)}', fontsize=11)
axes[1].axis('off')
plt.tight_layout()
plt.show()